In [ ]:
import numpy as np
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from typing import Tuple, List, Dict

# --- 1. Layers & Activation Functions ---

class Linear:
    """
    A fully connected (dense) layer.
    """
    def __init__(self, input_dim: int, output_dim: int):
        # He Initialization (Kaiming Init) - Best for ReLU
        scale = np.sqrt(2.0 / input_dim)
        self.W = np.random.randn(output_dim, input_dim) * scale
        self.b = np.zeros((output_dim, 1))
        
        # Gradients
        self.dW = None
        self.db = None
        
        # Cache for backward pass
        self.A_prev = None
        self.Z = None

    def forward(self, A_prev: np.ndarray) -> np.ndarray:
        """
        Z = W @ A_prev + b
        Stores A_prev and Z for backprop.
        """
        self.A_prev = A_prev
        self.Z = np.dot(self.W, A_prev) + self.b
        return self.Z

    def backward(self, dZ: np.ndarray) -> np.ndarray:
        """
        Calculates dW, db, and dA_prev.
        """
        m = self.A_prev.shape[1]  # Batch size
        
        # 1. Gradient of Cost w.r.t. Weights (dW)
        # Using the stored input activation A_prev
        self.dW = (1 / m) * np.dot(dZ, self.A_prev.T)
        
        # 2. Gradient of Cost w.r.t. Biases (db)
        self.db = (1 / m) * np.sum(dZ, axis=1, keepdims=True)
        
        # 3. Gradient of Cost w.r.t. Previous Activation (dA_prev)
        # The "Baton" passed to the previous layer
        dA_prev = np.dot(self.W.T, dZ)
        
        return dA_prev

    def update_params(self, lr: float):
        self.W -= lr * self.dW
        self.b -= lr * self.db


class ReLU:
    """
    Rectified Linear Unit Activation.
    """
    def __init__(self):
        self.Z = None

    def forward(self, Z: np.ndarray) -> np.ndarray:
        self.Z = Z
        return np.maximum(0, Z)

    def backward(self, dA: np.ndarray) -> np.ndarray:
        """
        dZ = dA * g'(Z)
        g'(Z) is 1 where Z > 0, else 0.
        """
        dZ = np.array(dA, copy=True)
        dZ[self.Z <= 0] = 0
        return dZ


class SoftmaxCrossEntropy:
    """
    Combines Softmax activation and Cross Entropy Loss for numerical stability.
    """
    def __init__(self):
        self.A_out = None  # Predicted probabilities
        self.Y = None      # True labels (One-hot)

    def forward(self, Z: np.ndarray, Y_true: np.ndarray) -> float:
        """
        Computes the loss and stores predictions for backprop.
        """
        # Shift Z for numerical stability (exp(large_num) causes overflow)
        shift_Z = Z - np.max(Z, axis=0, keepdims=True)
        exp_Z = np.exp(shift_Z)
        self.A_out = exp_Z / np.sum(exp_Z, axis=0, keepdims=True)
        self.Y = Y_true
        
        # Compute Cross Entropy Loss
        # L = -sum(y * log(a))
        m = Y_true.shape[1]
        loss = -np.sum(Y_true * np.log(self.A_out + 1e-9)) / m
        return loss

    def backward(self) -> np.ndarray:
        """
        The gradient of (Softmax + CrossEntropy) simplifies elegantly to (A - Y).
        This serves as dZ for the last layer.
        """
        return self.A_out - self.Y


# --- 2. The Model ---

class MLPManual:
    def __init__(self, input_size: int, hidden_size: int, output_size: int):
        self.layer1 = Linear(input_size, hidden_size)
        self.relu = ReLU()
        self.layer2 = Linear(hidden_size, output_size)
        self.criterion = SoftmaxCrossEntropy()

    def train_step(self, X: np.ndarray, Y: np.ndarray, lr: float) -> float:
        # --- Forward Pass ---
        Z1 = self.layer1.forward(X)
        A1 = self.relu.forward(Z1)
        Z2 = self.layer2.forward(A1)
        loss = self.criterion.forward(Z2, Y)

        # --- Backward Pass (Backpropagation) ---
        # 1. Start chain at Output (dZ_output = A - Y)
        dZ2 = self.criterion.backward()
        
        # 2. Backprop through Layer 2 to get dA1
        dA1 = self.layer2.backward(dZ2)
        
        # 3. Backprop through Activation (ReLU) to get dZ1
        dZ1 = self.relu.backward(dA1)
        
        # 4. Backprop through Layer 1 to get dW1, db1 (and dA0, which we ignore)
        _ = self.layer1.backward(dZ1)

        # --- Optimization ---
        self.layer1.update_params(lr)
        self.layer2.update_params(lr)

        return loss

    def predict(self, X: np.ndarray) -> np.ndarray:
        Z1 = self.layer1.forward(X)
        A1 = self.relu.forward(Z1)
        Z2 = self.layer2.forward(A1)
        # Softmax logic
        exp_Z = np.exp(Z2 - np.max(Z2, axis=0, keepdims=True))
        probs = exp_Z / np.sum(exp_Z, axis=0, keepdims=True)
        return np.argmax(probs, axis=0)

# --- 3. Execution Driver ---

def run_manual_demo():
    print("\n--- Running Manual NumPy Implementation ---")
    
    # 1. Load Data
    digits = load_digits()
    X = digits.data.T  # Shape: (64, m)
    y = digits.target
    
    # One-hot encode targets
    encoder = OneHotEncoder(sparse_output=False)
    Y_onehot = encoder.fit_transform(y.reshape(-1, 1)).T # Shape: (10, m)
    
    # Normalize Inputs
    X = X / 16.0 

    # Split
    split_idx = int(X.shape[1] * 0.8)
    X_train, X_test = X[:, :split_idx], X[:, split_idx:]
    Y_train, Y_test = Y_onehot[:, :split_idx], Y_onehot[:, split_idx:]
    y_test_labels = y[split_idx:]

    # 2. Init Model
    model = MLPManual(input_size=64, hidden_size=128, output_size=10)
    
    # 3. Training Loop
    epochs = 1000
    learning_rate = 0.01
    
    for i in range(epochs):
        loss = model.train_step(X_train, Y_train, learning_rate)
        if i % 200 == 0:
            print(f"Epoch {i}: Loss {loss:.4f}")

    # 4. Evaluation
    predictions = model.predict(X_test)
    accuracy = np.mean(predictions == y_test_labels)
    print(f"Manual Model Accuracy: {accuracy * 100:.2f}%")

run_manual_demo()


--- Running Manual NumPy Implementation ---
Epoch 0: Loss 2.5174
Epoch 200: Loss 2.5152
Epoch 400: Loss 2.5129
Epoch 600: Loss 2.5107
Epoch 800: Loss 2.5084
Manual Model Accuracy: 4.72%
